# Build an LLM chatbox — classroom workshop

**Duration:** 45–60 minutes · **Level:** beginner-friendly · **Default mode:** offline

You will inspect chat messages, improve a prompt, add conversation memory, launch an interactive chatbox, and learn how to switch the same interface to an internal or cloud model.

> Run cells from top to bottom. The committed outputs use the deterministic demo provider, so this notebook is safe to preview in GitHub and VS Code without credentials.

### Lesson map

~~~text
Notebook chatbox → ChatSession → provider configuration
                                  ├─ demo (offline)
                                  ├─ local/internal /v1 endpoint
                                  └─ cloud /v1 endpoint
~~~

Each numbered section is independent enough to pause for discussion or assign as a short task.

## 0. Setup

The project starts in demo mode. load_environment() reads .env if one exists, but never replaces a value already supplied as a Codespaces secret.

In [1]:
from pprint import pprint

from llm_workshop import (
    ChatSession,
    build_chat_widget,
    complete,
    load_config,
    load_environment,
)

load_environment()
config = load_config()
print(f"Ready: {config.label}")

Ready: demo / offline-demo


## 1. Messages

A chat request is a list of messages. The system message sets behavior; the user message carries the current task.

**Task 1 (5 min):** Change the user question below. Predict what the demo reply will mention before running the next cell.

In [2]:
messages = [
    {"role": "system", "content": "You are a concise classroom tutor."},
    {"role": "user", "content": "Explain why clear instructions help an LLM."},
]
pprint(messages)

[{'content': 'You are a concise classroom tutor.', 'role': 'system'},
 {'content': 'Explain why clear instructions help an LLM.', 'role': 'user'}]


In [3]:
reply = complete(messages, config)
print(reply)

Demo assistant (turn 1): I received ‘Explain why clear instructions help an LLM.’. A live model would answer here using the same chat history. Try making the request more specific by adding a goal, context, and output format.


> **Checkpoint:** Demo mode does not pretend to be intelligent. It proves that the notebook, message flow, and UI work without making a network request.

## 2. Prompt design

Useful prompts usually make four things explicit: **goal**, **context**, **constraints**, and **output format**.

**Task 2 (8 min):** Change one field below, rerun the cell, and compare the request with the simpler prompt from section 1.

In [4]:
goal = "Explain why language models can produce confident errors"
context = "The reader is new to machine learning"
constraints = "Use plain language and no more than 80 words"
output_format = "Return three bullet points"

structured_prompt = f"""
Goal: {goal}
Context: {context}
Constraints: {constraints}
Output format: {output_format}
""".strip()

print(structured_prompt)
print("\n--- assistant ---")
print(complete([messages[0], {"role": "user", "content": structured_prompt}], config))

Goal: Explain why language models can produce confident errors
Context: The reader is new to machine learning
Constraints: Use plain language and no more than 80 words
Output format: Return three bullet points

--- assistant ---
Demo assistant (turn 1): I received ‘Goal: Explain why language models can produce confident errors Context: The reader is new to machine learning Constraint’. A live model would answer here using the same chat history. Try making the request more specific by adding a goal, context, and output format.


**Extension:** When connected to a live model, test a vague and structured version of the same request. Decide what “better” means before comparing them.

## 3. Conversation memory

APIs are normally stateless. A chat application creates the feeling of memory by sending relevant earlier messages again.

**Task 3 (8 min):** Run two turns, then inspect exactly what the next request would contain.

In [5]:
session = ChatSession(
    config=config,
    system_prompt="You are a patient tutor. Ask for clarification when needed.",
)

print(session.send("Help me plan a short lesson about prompts."))
print()
print(session.send("Make the lesson suitable for pairs."))

Demo assistant (turn 1): I received ‘Help me plan a short lesson about prompts.’. A live model would answer here using the same chat history. Try making the request more specific by adding a goal, context, and output format.

Demo assistant (turn 2): I received ‘Make the lesson suitable for pairs.’. A live model would answer here using the same chat history. Try making the request more specific by adding a goal, context, and output format.


In [6]:
pprint(session.messages)

[{'content': 'You are a patient tutor. Ask for clarification when needed.',
  'role': 'system'},
 {'content': 'Help me plan a short lesson about prompts.', 'role': 'user'},
 {'content': 'Demo assistant (turn 1): I received ‘Help me plan a short lesson '
             'about prompts.’. A live model would answer here using the same '
             'chat history. Try making the request more specific by adding a '
             'goal, context, and output format.',
  'role': 'assistant'},
 {'content': 'Make the lesson suitable for pairs.', 'role': 'user'},
 {'content': 'Demo assistant (turn 2): I received ‘Make the lesson suitable '
             'for pairs.’. A live model would answer here using the same chat '
             'history. Try making the request more specific by adding a goal, '
             'context, and output format.',
  'role': 'assistant'}]


**Discuss:** What should happen when a conversation becomes too long? Options include trimming old turns, summarizing them, or retrieving only relevant context.

## 4. Interactive chatbox

The next cell creates a small ipywidgets interface. Type a message and choose **Send**. In demo mode, responses are instant and offline. Use **Reset** to clear only this widget's in-memory history.

**Task 4 (10 min):** Send two related messages and inspect chat_session.messages afterward.

In [7]:
chat_session = ChatSession(config=config)
chatbox = build_chat_widget(chat_session)
chatbox

In [8]:
# Rerun this after using the chatbox to inspect its state.
pprint(chat_session.messages)

[{'content': 'You are a concise and helpful classroom assistant.',
  'role': 'system'}]


## 5. Connect a live model

The notebook code does not change when the model location changes. Copy .env.example to .env, fill one provider block, and set LLM_PROVIDER. Never paste a real key into a notebook cell.

| LLM_PROVIDER | Use case | Required variables |
|---|---|---|
| demo | Offline classroom work | none |
| local | Ollama, vLLM, or another internal OpenAI-compatible server | LOCAL_LLM_BASE_URL, LOCAL_LLM_API_KEY, LOCAL_LLM_MODEL |
| classroom | Shared internal gateway | CLASSROOM_BASE_URL, CLASSROOM_API_KEY, CLASSROOM_MODEL |
| openai or cloud | OpenAI cloud | OPENAI_API_KEY, OPENAI_MODEL |
| custom | Another OpenAI-compatible cloud | CUSTOM_LLM_BASE_URL, CUSTOM_LLM_API_KEY, CUSTOM_LLM_MODEL |

After editing .env, **restart the kernel** and run from section 0. Codespaces secrets are exposed as environment variables and take precedence over .env.

In [9]:
# This safe check displays connection metadata, never the API key.
print({
    "provider": config.provider,
    "model": config.model,
    "base_url": config.base_url or "provider default",
    "credential_loaded": bool(config.api_key),
})

{'provider': 'demo', 'model': 'offline-demo', 'base_url': 'provider default', 'credential_loaded': False}


### Live connection challenge

1. Ask the instructor which provider and model to use.
2. Store credentials in .env or Codespaces secrets.
3. Restart the kernel and confirm section 0 no longer says demo / offline-demo.
4. Send a low-risk test prompt before adding private classroom material.

> A Codespace cannot reach a service bound only to localhost on your own laptop. Use a service reachable from the Codespace, or run the notebook locally for that endpoint.

## 6. Reflection

**Exit ticket (5 min):**

1. Where is conversation memory stored in this notebook?
2. Which setting changes the model location without changing the chatbox?
3. Why are committed demo outputs safer than committed live outputs?
4. What would you add before using this interface with real users?

**Suggested next iteration:** add streaming responses, a token/cost display, or a small evaluation set. Keep each change in its own Git commit so students can compare versions.